# Synergy prediction using bulk RNA-seq data

## Data loading

In [4]:
import pandas as pd
import numpy as np
import os
import functools
from functools import reduce

Load the l2fc (compared to time zero) data and the CFU data for each condition.

In [ ]:
import pandas as pd
import numpy as np
import os
import functools
from functools import reduce

def clean_csv_name(name):
    """
    Function to clean a csv name into a readable sample ID

    Args:
        name : ex: "T4-wt1AMX1hr-T4-wtNDC0hr-DGE-results"
    
    Returns:
        cleaned : ex: "1AMX1hr"
    """
    # Find index of . (.bam is at end of sample name) and remove filetag
    filetag_start_idx = name.rfind(".")
    cleaned = name[:filetag_start_idx]

    # Remove comparison ID and header
    tag1 = "-T4-wtNDC0hr-DGE-results"
    tag2 = "T4-wt"
    cleaned = cleaned.replace(tag1, "")
    cleaned = cleaned.replace(tag2, "")

    return cleaned

def read_l2fc_as_df(data_dir):
    """
    Function to extract all l2fc values from a directory of DGE results for various conditions.
    The output will be a list of dataframes, where each DGE result is stored as a dataframe with gene IDs on index, and 1 column of l2fc values

    Args:
        data_dir : path to folder containing DGE results

    Returns:
        l2fc_df_list : list of dataframes storing l2fc results
        names        : sample IDs corresponding to each dataframe
    """
    all_files = os.listdir(data_dir)
    files = [
        f for f in all_files
        if f.endswith(".csv") # select DGE results
        and "NDC0hr" in f # restrict to only time zero comparisons
            ]

    if len(files) == 0: 
        raise FileNotFoundError(f"No matching DGE files found in {data_dir}")

    # Sort and specify path to data files
    files = sorted(files)

    # Get sample IDs
    ids = [clean_csv_name(f) for f in files]

    # Convert csvs to dataframes
    filenames = ["".join([data_dir, "/" , f]) for f in files]
    l2fc_df_list = [pd.read_table(filenames[i], 
                                  sep = ",", 
                                  header = 0, 
                                  index_col = 1)["log2FoldChange"].rename(ids[i]) # Genes
                                  for i in range(len(filenames))]
    
    # Rename column in 
    
    # Rename the column to sample ID

    # Get corresponding sample IDs
    ids = [clean_csv_name(f) for f in files]

    return l2fc_df_list, ids

def bind_l2fc_data(l2fc_df_list, ids):
    """
    Function to take a list of l2fc DEG dataframes, then bind all into 1 dataframe
    Args: 
        l2fc_df_list : list of l2fc dataframes
        ids          : corresponding sample IDs

    Returns:
        all_l2fc [N,G] : dataframe with all feature values (N samples on row, G genes on column)
    """
    all_l2fc = reduce(lambda df1, df2 :
                      pd.merge(df1, df2, 
                               left_index = True, 
                               right_index = True, 
                               how = "outer"),
                        l2fc_df_list)

    # Tranpose to get genes on columns
    all_l2fc = all_l2fc.T

    return all_l2fc


def read_avg_cfus(folder_path):
    """
    Function to extract CFUs into a dataframe with conditions on rows, 1 CFU column
    Args:
        folder_path : path to folder containing CFUs (same format as /all_cfus)

    Returns:
        all_avg_cfus [N,1] : df with condition names as index, 1 column of avg CFUs across triplicates (N = # samples)
    """

    # Get files
    files = os.listdir(folder_path)
    
    # Select CSV files
    cfu_files = [csv for csv in files if ".csv" in csv]
    cfu_files = sorted(cfu_files)
    cfu_files = ["".join([folder_path, "/", csv]) for csv in files]
    
    # Load each file as a dataframe
    cfu_dfs = [pd.read_table(csv, sep = ",", header = 0) for csv in cfu_files] 

    # change all of this to calculate average CFU for each condition
    for i, df in enumerate(cfu_dfs):

        # Remove the triplicate column
        df = df.drop(columns = "Triplicates")

        # Calculate mean of three triplicates and store
        means = df.mean()
        cfu_dfs[i] = means
    
    # Concat
    all_avg_cfus = pd.concat(cfu_dfs, axis = 0) # series
    all_avg_cfus = all_avg_cfus.rename("CFU").to_frame()
    
    return all_avg_cfus


def bind_all_data(feature_df, cfu_df):
    """
    Function bind TPM and cfu dfs
    Args:
        featurem_df [N,G] : Dataframe of TPMs, N = # samples, G = # genes, labels on index
        cfu_df [N,1]      : Dataframe of CFUs, labels on index

    Returns:
        data_df : [N, G+1] : Dataframe of all TPMs and CFUs as last column
    """
    # Right join so that CFUs exist
    data_df = pd.merge(feature_df, cfu_df, left_index = True, right_index = True, how = "right")

    return data_df


def get_l2fc_and_cfu_data(l2fc_dir, cfu_dir):
    
    



# note: make sure that the sample IDs from the DGE matches the IDs for the CFU counts


SyntaxError: expected ':' (2157589324.py, line 149)

Run the data extraction pipeline.

In [ ]:
cfu_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/all_cfus"
l2fc_dir = "C:/Users/eddyk/OneDrive/Documents/vanopijnen_lab/dge_timezero"

,CFU
NDC1hr,7.933333e+08
14CEF1hr,4.800000e+08
13CEF1hr,4.666667e+08
12CEF1hr,2.266667e+08
34CEF1hr,1.600000e+08
...,...
12CEF34RIF4hr,4.600000e+07
34CEF14RIF4hr,6.133333e+07
34CEF13RIF4hr,3.133333e+07
34CEF12RIF4hr,2.866667e+07


Now, we can construct a metadata matrix that we will bind to the data. We will eventually have to calculate synergy scores and transcriptional interaction scores for all genes, so the annotated metatdata will help us match the approriate columns to calculate these.